# 数据预处理 —— 《飞驰人生3》短评数据清洗

**依赖库**：pandas、jieba  
**输入文件**：飞驰人生3.csv  
**输出文件**：飞驰人生3_cleaned.csv

## 0. 导入依赖库

In [1]:
import pandas as pd
import re
import jieba
from collections import Counter

d:\python\venv\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. 配置区

In [2]:
# 高赞阈值：votes_count 超过此值视为高赞评论
HIGH_VOTE_THRESHOLD = 100

# 最短有效评论字数（清洗后）
MIN_COMMENT_LENGTH = 3

# 电影专有词典（帮助 jieba 正确切分）
CUSTOM_WORDS = [
    "飞驰人生", "张驰", "沈腾", "韩寒", "孙宇强", "记星",
    "林臻东", "叶经理", "魏翔", "沙溢", "黄景瑜", "尹正",
    "沐尘100", "拉力赛", "赛车手", "换胎", "路书", "戈壁",
    "砂石路", "柏油路", "IMAX", "AI", "中速天梯",
]

# 停用词列表（常见无意义词）
STOPWORDS = {
    "的", "了", "在", "是", "我", "有", "和", "就", "不", "人",
    "都", "一", "一个", "上", "也", "很", "到", "说", "要", "去",
    "你", "会", "着", "没有", "看", "好", "这", "那", "但", "而",
    "与", "或", "被", "把", "让", "从", "对", "为", "以", "及",
    "等", "中", "之", "其", "于", "如", "则", "虽", "因", "所",
    "啊", "吧", "呢", "嗯", "哦", "哈", "嘛", "呀", "哇", "哎",
    "这个", "那个", "这种", "那种", "这部", "那部", "真的", "感觉",
    "觉得", "还是", "已经", "可以", "应该", "只是", "但是", "所以",
    "因为", "虽然", "如果", "不过", "然后", "还有", "而且", "并且",
    "电影", "影片", "片子", "观众", "看完", "看了", "看过",
}

## 2. 第一步：数据加载与初步检查

In [3]:
input_file  = "../1.爬虫/飞驰人生3.csv"
output_file = "飞驰人生3_cleaned.csv"

# 读取 CSV
df = pd.read_csv(input_file, encoding='utf-8-sig')

print(f"✅ 数据加载成功！")
print(f"   数据行数：{len(df)}")
print(f"   数据列数：{len(df.columns)}")

# 显示列名和数据类型
print(f"\n📋 数据字段信息：")
print(df.dtypes)

# 显示前 5 条数据
print(f"\n📊 数据样例（前5条）：")
df.head()

✅ 数据加载成功！
   数据行数：1200
   数据列数：8

📋 数据字段信息：
movie_name        object
comment_type       int64
comment_text      object
rating             int64
votes_count        int64
created_at        object
comment_length     int64
source_page        int64
dtype: object

📊 数据样例（前5条）： 


,movie_name,comment_type,comment_text,rating,votes_count,created_at,comment_length,source_page
0,飞驰人生3,1,目前系列最佳，非常纯粹的赛车电影。文戏精炼克制，讽刺意味却丝毫未减。赛车戏占比大幅提高，镜头...,5,4558,2026-02-17 12:10:51,113,1
1,飞驰人生3,1,韩寒写了部真正的中国侠客故事，事了拂衣去，干干净净赢。,4,4214,2026-02-17 11:11:34,27,1
2,飞驰人生3,1,这次赛车戏量大管饱，三部之最，且在前两部的基础上做了升级，作为洲际比赛的沐尘100拉力赛，比...,4,2723,2026-02-17 11:13:31,178,1
3,飞驰人生3,1,完全超乎我的预料，简直屌炸天！,5,2526,2026-02-17 11:09:03,15,1
4,飞驰人生3,1,这个系列每一步都走得很扎实，第一部讲求生，第二部讲求胜，第三部整个格局和立意都打开，在具备高...,4,1710,2026-02-17 11:14:48,259,1


In [4]:
# 检查缺失值
print("🔍 缺失值统计：")
missing_stats = df.isnull().sum()
print(missing_stats)
print(f"   总缺失值：{missing_stats.sum()}")

# 基本统计信息
print(f"\n📈 数值字段统计：")
df.describe()

🔍 缺失值统计：
movie_name        0
comment_type      0
comment_text      0
rating            0
votes_count       0
created_at        0
comment_length    0
source_page       0
dtype: int64
   总缺失值：0

📈 数值字段统计：


,comment_type,rating,votes_count,comment_length,source_page
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,0.000000,2.989167,67.056667,75.012500,10.500000
std,0.816837,1.126991,402.363163,80.746604,5.768685
min,-1.000000,1.000000,0.000000,1.000000,1.000000
25%,-1.000000,2.000000,0.000000,19.000000,5.750000
50%,0.000000,3.000000,2.000000,44.000000,10.500000
75%,1.000000,4.000000,12.000000,101.000000,15.250000
max,1.000000,5.000000,7649.000000,350.000000,20.000000


## 3. 第二步：数据清洗（去重 + 删除无效记录）

In [5]:
original_count = len(df)

# 2.1 去重：基于"评论文本+发布时间"组合
df = df.drop_duplicates(subset=["comment_text", "created_at"], keep="first")
print(f"✅ 去重后：{len(df)} 条（删除 {original_count - len(df)} 条重复）")

# 2.2 删除评论文本为空的记录
df = df[df["comment_text"].notna() & (df["comment_text"].str.strip() != "")]
print(f"✅ 删除空评论后：{len(df)} 条")

# 2.3 删除评分缺失的记录（rating 为 NaN）
df = df[df["rating"].notna()]
print(f"✅ 删除评分缺失后：{len(df)} 条")

# 2.4 删除发布时间缺失的记录
df = df[df["created_at"].notna() & (df["created_at"].str.strip() != "")]
print(f"✅ 删除时间缺失后：{len(df)} 条")

# 2.5 过滤极短文本
df = df[df["comment_length"] >= MIN_COMMENT_LENGTH]
print(f"✅ 过滤极短文本（<{MIN_COMMENT_LENGTH}字）后：{len(df)} 条")

print(f"\n📊 清洗小结：原始 {original_count} 条 → 保留 {len(df)} 条，删除 {original_count - len(df)} 条")
df = df.reset_index(drop=True)

✅ 去重后：1200 条（删除 0 条重复）
✅ 删除空评论后：1200 条
✅ 删除评分缺失后：1200 条
✅ 删除时间缺失后：1200 条
✅ 过滤极短文本（<3字）后：1190 条

📊 清洗小结：原始 1200 条 → 保留 1190 条，删除 10 条


## 4. 第三步：文本清洗（正则去噪）

In [6]:
def clean_text(text):
    """
    对单条评论文本进行正则清洗
    """
    # 去除 emoji（Unicode 表情符号范围）
    emoji_pattern = re.compile(
        "[\U00010000-\U0010ffff"
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\u2600-\u26FF\u2700-\u27BF]+",
        flags=re.UNICODE
    )
    text = emoji_pattern.sub("", text)

    # 去除 HTML 标签
    text = re.sub(r"<[^>]+>", "", text)

    # 只保留中文、英文字母、数字、常用标点
    text = re.sub(r"[^\u4e00-\u9fa5a-zA-Z0-9，。！？、；：\u201c\u201d\u2018\u2019（）【】…—\s]", "", text)

    # 合并多余空白（包括换行）
    text = re.sub(r"\s+", " ", text).strip()

    return text


# 对 comment_text 列批量清洗
df["comment_text"] = df["comment_text"].apply(clean_text)

# 重新计算清洗后的评论字数
df["comment_length"] = df["comment_text"].str.len()

# 再次过滤清洗后变为极短的文本
before = len(df)
df = df[df["comment_length"] >= MIN_COMMENT_LENGTH]
print(f"✅ 文本清洗完成，清洗后再次过滤极短文本，删除 {before - len(df)} 条")
print(f"   当前数据量：{len(df)} 条")
df = df.reset_index(drop=True)

✅ 文本清洗完成，清洗后再次过滤极短文本，删除 4 条
   当前数据量：1186 条


## 5. 第四步：jieba 分词

In [7]:
# 加载自定义词典
for word in CUSTOM_WORDS:
    jieba.add_word(word)
print("✅ 自定义词典加载完成")


def tokenize(text):
    """
    对单条文本进行分词，去除停用词，返回空格分隔的词语字符串
    """
    words = jieba.cut(text, cut_all=False)
    filtered = [w for w in words if w.strip() and w not in STOPWORDS and len(w) > 1]
    return " ".join(filtered)


df["tokens"] = df["comment_text"].apply(tokenize)
print(f"✅ 分词完成，共处理 {len(df)} 条评论")

# 展示前 3 条分词结果
print("\n📋 分词样例（前3条）：")
for i, row in df.head(3).iterrows():
    print(f"  原文：{row['comment_text'][:40]}...")
    print(f"  分词：{row['tokens'][:60]}...")
    print()

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\SPY\AppData\Local\Temp\jieba.cache
Loading model cost 0.311 seconds.
Prefix dict has been built successfully.


✅ 自定义词典加载完成
✅ 分词完成，共处理 1186 条评论

📋 分词样例（前3条）：
  原文：目前系列最佳，非常纯粹的赛车电影。文戏精炼克制，讽刺意味却丝毫未减。赛车戏占比大...
  分词：目前 系列 最佳 非常 纯粹 赛车 文戏 精炼 克制 讽刺 意味 丝毫 未减 赛车 大幅提高 镜头 很帅 全程 高燃 加...

  原文：韩寒写了部真正的中国侠客故事，事了拂衣去，干干净净赢。...
  分词：韩寒 真正 中国 侠客 故事 拂衣 干干净净...

  原文：这次赛车戏量大管饱，三部之最，且在前两部的基础上做了升级，作为洲际比赛的沐尘10...
  分词：这次 赛车 戏量 管饱 三部 两部 基础 升级 作为 洲际 比赛 沐尘100 拉力赛 比巴音 布鲁克 难度 更高 柏油路...



## 6. 第五步：构造辅助变量

In [8]:
# 5.1 解析 created_at 为 datetime
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

# 5.2 提取评论日期（年-月-日）
df["comment_date"] = df["created_at"].dt.date


# 5.3 提取评论时段
def get_time_period(dt):
    if pd.isna(dt):
        return "未知"
    hour = dt.hour
    if 6 <= hour < 12:
        return "上午"
    elif 12 <= hour < 18:
        return "下午"
    elif 18 <= hour < 24:
        return "晚上"
    else:
        return "深夜"


df["time_period"] = df["created_at"].apply(get_time_period)

# 5.4 是否高赞（votes_count > HIGH_VOTE_THRESHOLD）
df["is_high_vote"] = (df["votes_count"] > HIGH_VOTE_THRESHOLD).astype(int)

high_vote_count = df["is_high_vote"].sum()
print(f"✅ 辅助变量构造完成")
print(f"   高赞评论（>{HIGH_VOTE_THRESHOLD}赞）：{high_vote_count} 条 / {len(df)} 条")
print(f"   时段分布：")
print(df["time_period"].value_counts())

✅ 辅助变量构造完成
   高赞评论（>100赞）：86 条 / 1186 条
   时段分布：
time_period
晚上    478
下午    433
上午    148
深夜    127
Name: count, dtype: int64


## 7. 第六步：保存清洗后数据

In [9]:
fieldnames = [
    "movie_name", "comment_type", "comment_text", "tokens",
    "rating", "votes_count", "is_high_vote",
    "created_at", "comment_date", "time_period",
    "comment_length", "source_page",
]
df = df[fieldnames]
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✅ 已保存至：{output_file}")
print(f"   最终数据量：{len(df)} 条")

type_map = {1: "好评", 0: "中评", -1: "差评"}
print(f"\n📊 各评价类型分布：")
for val, label in type_map.items():
    count = (df["comment_type"] == val).sum()
    print(f"   {label}（{val}）：{count} 条")

✅ 已保存至：飞驰人生3_cleaned.csv
   最终数据量：1186 条

📊 各评价类型分布：
   好评（1）：399 条
   中评（0）：396 条
   差评（-1）：391 条


## 8. 预处理完成

至此，数据预处理全部完成。清洗后的数据已保存至 `飞驰人生3_cleaned.csv`，可用于后续的情感分析、词云可视化等任务。

In [10]:
print(f"🎉 预处理完成！原始 {original_count} 条 → 最终 {len(df)} 条")
df.head()

🎉 预处理完成！原始 1200 条 → 最终 1186 条


,movie_name,comment_type,comment_text,tokens,rating,votes_count,is_high_vote,created_at,comment_date,time_period,comment_length,source_page
0,飞驰人生3,1,目前系列最佳，非常纯粹的赛车电影。文戏精炼克制，讽刺意味却丝毫未减。赛车戏占比大幅提高，镜头...,目前 系列 最佳 非常 纯粹 赛车 文戏 精炼 克制 讽刺 意味 丝毫 未减 赛车 大幅提高...,5,4558,1,2026-02-17 12:10:51,2026-02-17,下午,113,1
1,飞驰人生3,1,韩寒写了部真正的中国侠客故事，事了拂衣去，干干净净赢。,韩寒 真正 中国 侠客 故事 拂衣 干干净净,4,4214,1,2026-02-17 11:11:34,2026-02-17,上午,27,1
2,飞驰人生3,1,这次赛车戏量大管饱，三部之最，且在前两部的基础上做了升级，作为洲际比赛的沐尘100拉力赛，比...,这次 赛车 戏量 管饱 三部 两部 基础 升级 作为 洲际 比赛 沐尘100 拉力赛 比巴音...,4,2723,1,2026-02-17 11:13:31,2026-02-17,上午,175,1
3,飞驰人生3,1,完全超乎我的预料，简直屌炸天！,完全 超乎 预料 简直 炸天,5,2526,1,2026-02-17 11:09:03,2026-02-17,上午,15,1
4,飞驰人生3,1,这个系列每一步都走得很扎实，第一部讲求生，第二部讲求胜，第三部整个格局和立意都打开，在具备高...,系列 一步 扎实 第一部 讲求 第二部 讲求 第三部 整个 格局 立意 打开 具备 高度 赛...,4,1710,1,2026-02-17 11:14:48,2026-02-17,上午,258,1
